# Week 23, Day 3: JSON

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w023d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import json

data = {"name": "Ada", "age": 36, "langs": ["Python", "C"], "active": True, "boss": None}
text = json.dumps(data)
print(text)
print(type(text).__name__)
print(json.loads(text) == data)

In [ ]:
import json

data = {"city": "Lagos", "temps": [29.4, 31], "warning": None, "note": "Ẹ kú àárọ̀"}
print(json.dumps(data))
print(json.dumps(data, indent=2, sort_keys=True, ensure_ascii=False))
print(json.dumps({1: "one", "two": (2, 2)}))

In [ ]:
import json

for text in ['{"a": 1}', "{'a': 1}", '[1, 2,]', '{"a": 1} extra']:
    try:
        print(json.loads(text))
    except json.JSONDecodeError as e:
        print("invalid:", e.msg, "at position", e.pos)

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('settings.json', 'w', encoding='utf-8') as f:
    f.write('{\n  "theme": "dark",\n  "font_size": 14,\n  "recent": ["notes.txt", "todo.txt"]\n}\n')

In [ ]:
import json

with open("settings.json", encoding="utf-8") as f:
    settings = json.load(f)
print(settings["theme"], settings["recent"])

settings["font_size"] += 2
settings["recent"].insert(0, "ideas.txt")
with open("settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

with open("settings.json", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import json

response = json.loads('''{
  "city": "Lagos",
  "current": {"temp_c": 29.4, "humidity": 78},
  "forecast": [{"day": "Sat", "high": 31, "low": 25}, {"day": "Sun", "high": 30, "low": 24}]
}''')

print(response["current"]["temp_c"])
print([d["high"] for d in response["forecast"]])
print(response.get("warning", "no warning"))
print(response["current"].get("wind_kph"))
highest = max(response["forecast"], key=lambda d: d["high"])
print(f"Warmest: {highest['day']} at {highest['high']}°C")

In [ ]:
import json
from datetime import date

def to_json(value):
    if isinstance(value, date):
        return {"$date": value.isoformat()}
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"can't save {type(value).__name__}")

def from_json(obj):
    if set(obj) == {"$date"}:
        return date.fromisoformat(obj["$date"])
    return obj

event = {"title": "Launch", "on": date(2026, 10, 31), "tags": {"python", "atlas"}}
text = json.dumps(event, default=to_json)
print(text)
back = json.loads(text, object_hook=from_json)
print(back["on"], type(back["on"]).__name__, back["tags"])

> **Key idea.** JSON has objects, arrays, strings, numbers, true/false and null, which become `dict`, `list`, `str`, `int`/`float`, `bool` and `None`. `dumps`/`loads` convert strings; `dump`/`load` work with files (UTF-8). Keys are always strings, quotes always double. Use `indent=` for people, `.get` for missing fields, `JSONDecodeError` for bad input, and `default=` (with your own reader) for dates, sets and other types.

### Exercise: There and back

Predict what comes back after `json.loads(json.dumps(x))` for each value, then check. Set each variable to the value you get back.

In [ ]:
import json

a = ___   # from (1, 2)
b = ___   # from {1: "one"}
c = ___   # from {"ok": True, "none": None}
d = ___   # from 2.50

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 2] and type(a) is list\n    assert b == {\"1\": \"one\"}\n    assert c == {\"ok\": True, \"none\": None} and d == 2.5")

### Exercise: Read a response

`RESPONSE` holds JSON from a weather service. Write `summary(text)` that returns a string like `"Lagos: 29.4°C now, high 31°C tomorrow (Sat)"` using the city, the current temperature and the **first** forecast day; and `rainy_days(text)` returning the list of forecast days where `"rain_mm"` is more than 0 (some days don't have a `"rain_mm"` field at all: treat that as 0).

In [ ]:
import json

RESPONSE = '''{
  "city": "Lagos",
  "current": {"temp_c": 29.4, "humidity": 78},
  "forecast": [
    {"day": "Sat", "high": 31, "low": 25, "rain_mm": 0},
    {"day": "Sun", "high": 30, "low": 24, "rain_mm": 12.5},
    {"day": "Mon", "high": 29, "low": 24},
    {"day": "Tue", "high": 28, "low": 23, "rain_mm": 3}
  ]
}'''

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert summary(RESPONSE) == \"Lagos: 29.4\u00b0C now, high 31\u00b0C tomorrow (Sat)\"\n    assert rainy_days(RESPONSE) == [\"Sun\", \"Tue\"]")

### Exercise: Save and load a game

Write `save_game(path, state)` that writes the dictionary `state` to a JSON file (UTF-8, `indent=2`, `sort_keys=True`), and `load_game(path)` that reads it back, returning the default state `{"level": 1, "score": 0, "inventory": []}` if the file doesn't exist **or** isn't valid JSON. The checks look at the file's text, so the layout matters.

In [ ]:
import json

DEFAULT = {"level": 1, "score": 0, "inventory": []}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    reset_files()\n    assert load_game(\"save.json\") == {\"level\": 1, \"score\": 0, \"inventory\": []}\n    save_game(\"save.json\", {\"score\": 120, \"level\": 3, \"inventory\": [\"key\", \"map\"]})\n    assert read_file(\"save.json\") == '{\\n  \"inventory\": [\\n    \"key\",\\n    \"map\"\\n  ],\\n  \"level\": 3,\\n  \"score\": 120\\n}'\n    assert load_game(\"save.json\") == {\"level\": 3, \"score\": 120, \"inventory\": [\"key\", \"map\"]}\n    write_file(\"broken.json\", \"{not json\")\n    d = load_game(\"broken.json\")\n    assert d == {\"level\": 1, \"score\": 0, \"inventory\": []}\n    d[\"inventory\"].append(\"x\")\n    assert load_game(\"broken.json\")[\"inventory\"] == [], \"return a fresh copy of the default each time\"")

### Exercise: Flatten nested JSON

Spreadsheets want flat rows. Write `flatten(data, prefix="")` that turns nested dictionaries into one dictionary with dotted keys: `{"a": {"b": 1, "c": {"d": 2}}, "e": 3}` becomes `{"a.b": 1, "a.c.d": 2, "e": 3}`. Lists stay as they are. Then `unflatten(flat)` that does the reverse.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    nested = {\"a\": {\"b\": 1, \"c\": {\"d\": 2}}, \"e\": 3, \"tags\": [\"x\", \"y\"]}\n    flat = flatten(nested)\n    assert flat == {\"a.b\": 1, \"a.c.d\": 2, \"e\": 3, \"tags\": [\"x\", \"y\"]}\n    assert unflatten(flat) == nested\n    assert flatten({}) == {} and unflatten({}) == {}")

### Exercise: Dates in and out

Write `dumps_with_dates(data)` that converts data containing `date` and `datetime` values (anywhere inside) to JSON text, writing dates as ISO strings, and `loads_with_dates(text, fields)` that reads JSON back and converts the values of the keys named in `fields` (in any dictionary, at any depth) from ISO strings to `date` objects (use `date.fromisoformat`). Use `default=` and `object_hook=`.

In [ ]:
import json
from datetime import date, datetime

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import date\n    data = {\"title\": \"Launch\", \"on\": date(2026, 10, 31), \"tasks\": [{\"name\": \"Write\", \"due\": date(2026, 10, 20)}]}\n    text = dumps_with_dates(data)\n    assert '\"on\": \"2026-10-31\"' in text and '\"due\": \"2026-10-20\"' in text\n    back = loads_with_dates(text, {\"on\", \"due\"})\n    assert back == data\n    assert loads_with_dates('{\"on\": null}', {\"on\"}) == {\"on\": None}")

### Bug hunt: Not quite JSON

This text was written by hand and won't load: it has single quotes, a trailing comma and Python's `True`. Fix the text (not the code) so it prints `{'name': 'Ada', 'langs': ['Python', 'C'], 'active': True}`.

In [ ]:
import json

TEXT = "{'name': 'Ada', 'langs': ['Python', 'C',], 'active': True}"
print(json.loads(TEXT))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"valid JSON\"\n    expect_output(\"{'name': 'Ada', 'langs': ['Python', 'C'], 'active': True}\")\n    assert \"json.loads(TEXT)\" in source")

### Bug hunt: Keys come back as strings

This saves marks keyed by student number, and then looks up student `7` after loading, which crashes with `KeyError: 7`: JSON turned the keys into strings. Fix the lookup side (convert the keys back to `int` after loading) so it prints `82`.

In [ ]:
import json

marks = {7: 82, 12: 64}
text = json.dumps(marks)
loaded = json.loads(text)
print(loaded[7])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"82\")")

### Puzzle: Update a settings file

Arrange the lines so the program loads `settings.json`, turns on dark mode, and saves it again. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
import json

a = ___   # json.dumps([True, None, 1.0])
b = ___   # json.loads('{"x": [1, {"y": null}]}')["x"][1]["y"]
c = ___   # json.dumps({"b": 1, "a": 2}, sort_keys=True)
d = ___   # the type name of json.loads("3.0")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"[true, null, 1.0]\" and b is None and c == '{\"a\": 2, \"b\": 1}' and d == \"float\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: validate a JSON document

Before trusting JSON from outside, programs check it has the right shape. Write `validate(data, schema, path="$")` that checks `data` against a small schema and returns a list of error messages (empty if valid). A schema is either:

- a Python type (`str`, `int`, `float`, `bool`): the value must be of that type (an `int` is accepted where `float` is expected, but a `bool` is never accepted as an `int` or `float`);
- a list with one schema in it, `[schema]`: the value must be a list, and every item must match;
- a dictionary from key to schema: the value must be a dictionary with every key present and matching; a key ending in `?` is optional.

Messages look like `"$.forecast[1].high: expected int, got str"`, `"$.city: missing"` or `"$.current: expected object, got list"` (use `object` for dictionaries and `array` for lists).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("SCHEMA = {\"city\": str, \"current\": {\"temp_c\": float, \"raining?\": bool}, \"forecast\": [{\"day\": str, \"high\": int}]}\ndef test_valid():\n    \"valid\"\n    good = {\"city\": \"Lagos\", \"current\": {\"temp_c\": 29}, \"forecast\": [{\"day\": \"Sat\", \"high\": 31}]}\n    assert validate(good, SCHEMA) == []\ndef test_invalid():\n    \"errors\"\n    bad = {\"current\": [1, 2], \"forecast\": [{\"day\": \"Sat\", \"high\": 31}, {\"day\": \"Sun\", \"high\": \"30\"}, {\"high\": True}]}\n    assert validate(bad, SCHEMA) == [\n        \"$.city: missing\",\n        \"$.current: expected object, got array\",\n        \"$.forecast[1].high: expected int, got str\",\n        \"$.forecast[2].day: missing\",\n        \"$.forecast[2].high: expected int, got bool\",\n    ]\n    assert validate({\"current\": {\"temp_c\": 1.5, \"raining\": \"no\"}, \"city\": \"X\", \"forecast\": []}, SCHEMA) == [\"$.current.raining: expected bool, got str\"]")

## Warm-up

The JSON text looks almost like the dictionary, with three differences you can spot: `true` instead of `True`, `null` instead of `None`, and always double quotes. `dumps` makes text from Python data; `loads` turns it back, and you get an equal dictionary.

## What JSON looks like

**JSON** (JavaScript Object Notation) is a text format for data. It came from JavaScript but every language reads it, and it's the standard for web services and settings files. It has just six kinds of value:

| JSON | Example | Python |
|---|---|---|
| object | `{"name": "Ada", "age": 36}` | `dict` |
| array | `[1, 2, 3]` | `list` |
| string | `"hello"` | `str` |
| number | `36`, `-2.5`, `1e3` | `int` or `float` |
| true / false | `true` | `True` / `False` |
| null | `null` | `None` |

JSON's rules are strict, and this is where people trip:

- Strings and keys use **double** quotes only: `{'a': 1}` is not valid JSON.
- Object **keys must be strings**. Python allows `{1: "a"}`; `json.dumps` turns the key into `"1"`, and it comes back as a string.
- No trailing commas: `[1, 2,]` is invalid.
- No comments.
- No tuples, sets, dates or other types (tuples become arrays).

Here's a typical response from a weather web service:

```
{
  "city": "Lagos",
  "current": {"temp_c": 29.4, "humidity": 78, "raining": false},
  "forecast": [
    {"day": "Sat", "high": 31, "low": 25},
    {"day": "Sun", "high": 30, "low": 24}
  ],
  "warning": null
}
```

## Converting both ways

| Function | Converts |
|---|---|
| `json.dumps(data)` | Python data to a JSON **string** ("dump string") |
| `json.loads(text)` | a JSON string to Python data ("load string") |
| `json.dump(data, file)` | Python data into an open file |
| `json.load(file)` | an open file's JSON into Python data |

`dumps` takes options that make the output nicer:

- `indent=2` spreads it over lines, for people to read (and for clean Git diffs).
- `sort_keys=True` puts keys in order, so the same data always gives the same text.
- `ensure_ascii=False` keeps letters like "ẹ" as they are, instead of `ẹ` codes. Both are valid JSON; the second is safer for old systems, the first is readable.

**Invalid JSON** raises `json.JSONDecodeError`, which says where the problem is:

## Files

Save and load with `dump` and `load`, opening the file with an explicit encoding (JSON is UTF-8 by definition):

This is the simplest way to **save a program's state** between runs: a dictionary in, a dictionary out. Week 21's `snap` did exactly this.

## Nested data

Real JSON is nested: objects inside arrays inside objects. Navigate it step by step with `[]`, and use `.get` (with a default) for parts that may be missing, since web services often leave out empty fields:

When the structure is unfamiliar, print it with `json.dumps(data, indent=2)`, or look at the keys with `list(data)`, before writing code against it. Week 10's `nestview` figure is a good mental model.

## Types JSON doesn't have

Dates, sets, `Decimal`s and your own objects can't go into JSON directly: `json.dumps` raises `TypeError: Object of type date is not JSON serializable`. The fix has two halves:

- **Writing:** pass `default=`, a function that `dumps` calls for anything it doesn't know, which returns something it does (a string, list or dictionary).
- **Reading:** convert those values back yourself after loading, or pass `object_hook=`, a function called on every object (dictionary) as it's read.

The set came back as a list: JSON can't tell you it was a set, so your code has to know what each field means. Dates are usually stored as ISO strings (`"2026-10-31"`), as here.

## Practice

## Quiz

## Daily challenge

## Before you go

Many websites have free JSON you can look at in a browser: try `https://api.github.com/repos/python/cpython` and read the shape of what comes back. Tomorrow: CSV, the format of spreadsheets, which is simpler than JSON and full of surprises.